# PPL Evaluation Debug Notebook
This notebook is for debugging the perplexity evaluation pipeline based on `run_ppl_eval.py`.


In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset
from tqdm import tqdm
import argparse
import os
from utils import load_model_and_tokenizer, add_common_args
from palu.quant_utils import configure_latent_quantizer
from loguru import logger
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, Any
import pandas as pd

# Setup logger for notebook
logger.remove()
logger.add(lambda msg: print(msg, end=""), colorize=True, level="INFO")

print("Imports completed successfully!")


Imports completed successfully!


## Configuration


In [2]:
# Configuration - modify these as needed for debugging
CONFIG = {
    'model_name_or_path': '/home/xinj/palu/Meta-Llama-3-8B-Instruct_ratio-0.7_gs-4-fisher_uniform-rope_svd',
    'datasets': 'wikitext2',  # 'wikitext2' or 'c4'
    'seqlen': 2048,
    'device': 'cuda',
    'lt_bits': 16,
    'lt_group_size': 0,
    'lt_sym': False,
    'lt_clip_ratio': 1.0,
    'lt_hadamard': False,
    'use_flash_attn2': False,
    'verbose': True,
    'debug_samples': 5,  # Number of samples to debug in detail
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")


Configuration:
  model_name_or_path: /home/xinj/palu/Meta-Llama-3-8B-Instruct_ratio-0.7_gs-4-fisher_uniform-rope_svd
  datasets: wikitext2
  seqlen: 2048
  device: cuda
  lt_bits: 16
  lt_group_size: 0
  lt_sym: False
  lt_clip_ratio: 1.0
  lt_hadamard: False
  use_flash_attn2: False
  verbose: True
  debug_samples: 5


## Data Loading Functions


In [3]:
def get_ppl_eval_loaders(name, tokenizer, seqlen=2048):
    """Load evaluation datasets for perplexity calculation"""
    if "wikitext2" in name:
        print(f"Loading WikiText-2 dataset...")
        testdata = load_dataset(
            "Salesforce/wikitext",
            "wikitext-2-raw-v1",
            split="test",
        )
        print(f"Dataset loaded. Number of text samples: {len(testdata['text'])}")
        
        # Show some sample texts for debugging
        print("\nSample texts:")
        for i, text in enumerate(testdata["text"][:3]):
            if text.strip():  # Only show non-empty texts
                print(f"  Sample {i}: {text[:100]}...")
        
        testenc = tokenizer("\n\n".join(testdata["text"]), return_tensors="pt")
        print(f"Tokenized sequence length: {testenc.input_ids.shape[1]}")
        return testenc
        
    elif "c4" in name:
        print(f"Loading C4 dataset...")
        # Wrapper for tokenized input IDs
        class TokenizerWrapper:
            def __init__(self, input_ids):
                self.input_ids = input_ids
                
        valdata = load_dataset(
            "allenai/c4",
            data_files={"validation": "en/c4-validation.00000-of-00008.json.gz"},
            revision="607bd4c8450a42878aa9ddc051a65a055450ef87",
            split="validation",
        )
        print(f"Dataset loaded. Number of validation samples: {len(valdata)}")
        
        # Show some sample texts for debugging
        print("\nSample C4 texts:")
        for i in range(min(3, len(valdata))):
            print(f"  Sample {i}: {valdata[i]['text'][:100]}...")
        
        testenc = tokenizer(' '.join(valdata[:1100]['text']), return_tensors='pt')
        testenc = testenc.input_ids[:, :(256 * seqlen)]
        testenc = TokenizerWrapper(testenc)
        print(f"Tokenized sequence length: {testenc.input_ids.shape[1]}")
        return testenc
    else:
        raise NotImplementedError(f"Dataset {name} not implemented")

print("Data loading functions defined!")


Data loading functions defined!


## Model Loading and Inspection


In [4]:
# Load model and tokenizer
print(f"Loading model from: {CONFIG['model_name_or_path']}")
model, tokenizer = load_model_and_tokenizer(CONFIG['model_name_or_path'], use_flash_attn2=CONFIG['use_flash_attn2'])

print(f"Model loaded successfully!")
print(f"Model type: {type(model)}")
print(f"Model config keys: {list(model.config.to_dict().keys())}")
print(f"Tokenizer: {type(tokenizer)}")
print(f"Vocab size: {tokenizer.vocab_size}")

# Check model architecture - specifically attention modules
print("\nAttention modules:")
attention_count = 0
for name, module in model.named_modules():
    if 'self_attn' in name and len(name.split('.')) == 3:  # Top level attention modules
        print(f"  {name}: {type(module)}")
        attention_count += 1
        if attention_count >= 3:  # Show first 3
            break

# Check for rope_latent config
if hasattr(model.config, 'rope_latent'):
    print(f"\nModel config rope_latent: {model.config.rope_latent}")
else:
    print("\nModel config does NOT have rope_latent attribute")


Loading model from: /home/xinj/palu/Meta-Llama-3-8B-Instruct_ratio-0.7_gs-4-fisher_uniform-rope_svd


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully!
Model type: <class 'palu.model.svd_llama.modeling_palu_llama.PaluLlamaForCausalLM'>
Model config keys: ['vocab_size', 'max_position_embeddings', 'hidden_size', 'intermediate_size', 'num_hidden_layers', 'num_attention_heads', 'num_key_value_heads', 'hidden_act', 'initializer_range', 'rms_norm_eps', 'pretraining_tp', 'use_cache', 'rope_theta', 'rope_scaling', 'attention_bias', 'mlp_bias', 'return_dict', 'output_hidden_states', 'output_attentions', 'torchscript', 'torch_dtype', 'use_bfloat16', 'tf_legacy_loss', 'pruned_heads', 'tie_word_embeddings', 'chunk_size_feed_forward', 'is_encoder_decoder', 'is_decoder', 'cross_attention_hidden_size', 'add_cross_attention', 'tie_encoder_decoder', 'max_length', 'min_length', 'do_sample', 'early_stopping', 'num_beams', 'num_beam_groups', 'diversity_penalty', 'temperature', 'top_k', 'top_p', 'typical_p', 'repetition_penalty', 'length_penalty', 'no_repeat_ngram_size', 'encoder_no_repeat_ngram_size', 'bad_words_ids', 'num_retu

## PPL Evaluation Function with Debugging


In [ ]:
@torch.no_grad()
def eval_ppl_debug(model, tokenizer, testenc, seqlen=2048, device="cuda", max_samples=None, verbose=True):
    """Evaluate perplexity with detailed debugging information"""
    model = model.to(device)
    if isinstance(device, str):
        device = torch.device(device)

    nsamples = testenc.numel() // seqlen
    if max_samples is not None:
        nsamples = min(nsamples, max_samples)
        
    print(f"Evaluating on {nsamples} samples...")
    
    use_cache = model.config.use_cache
    model.config.use_cache = False
    model.eval()

    nlls = []
    detailed_stats = []

    for i in tqdm(range(nsamples), desc="Evaluating PPL"):
        batch = testenc[:, (i * seqlen) : ((i + 1) * seqlen)].to(device)
        
        # Forward pass
        outputs = model.model(batch)
        hidden_states = outputs[0]
        logits = model.lm_head(hidden_states)
        
        # Calculate loss
        shift_logits = logits[:, :-1, :]
        shift_labels = testenc[:, (i * seqlen) : ((i + 1) * seqlen)][:, 1:].to(device)
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(shift_logits.reshape(-1, shift_logits.size(-1)), shift_labels.reshape(-1))
        
        neg_log_likelihood = loss.float() * seqlen
        nlls.append(neg_log_likelihood)
        
        # Collect detailed statistics for debugging
        if verbose and i < CONFIG['debug_samples']:
            stats = {
                'sample_idx': i,
                'loss': loss.item(),
                'nll': neg_log_likelihood.item(),
                'logits_mean': logits.mean().item(),
                'logits_std': logits.std().item(),
                'logits_max': logits.max().item(),
                'logits_min': logits.min().item(),
                'hidden_states_norm': hidden_states.norm().item(),
            }
            detailed_stats.append(stats)
            
            if i < 2:  # Show first 2 samples in detail
                print(f"\nSample {i} details:")
                print(f"  Input text: {tokenizer.decode(batch[0, :50])}...")
                print(f"  Loss: {loss.item():.4f}")
                print(f"  NLL: {neg_log_likelihood.item():.4f}")
                print(f"  Logits shape: {logits.shape}")
                print(f"  Hidden states shape: {hidden_states.shape}")
            
    ppl = torch.exp(torch.stack(nlls).sum() / (len(nlls) * seqlen))
    model.config.use_cache = use_cache
    
    return ppl.item(), nlls, detailed_stats

print("PPL evaluation function defined!")


## Load Test Data and Run Evaluation


In [ ]:
# Load test data
dataset_name = CONFIG['datasets']
seqlen = CONFIG['seqlen']

cache_testloader = f"/tmp/{dataset_name}_testloader_{CONFIG['model_name_or_path'].replace('/', '_')}_all.cache"
print(f"Cache file: {cache_testloader}")

if os.path.exists(cache_testloader):
    print("Loading from cache...")
    testloader = torch.load(cache_testloader)
else:
    print("Creating new testloader...")
    testloader = get_ppl_eval_loaders(dataset_name, tokenizer, seqlen)
    print("Saving to cache...")
    torch.save(testloader, cache_testloader)

testenc = testloader.input_ids
nsamples = testenc.numel() // seqlen

print(f"\nDataset statistics:")
print(f"  Total tokens: {testenc.numel()}")
print(f"  Sequence length: {seqlen}")
print(f"  Number of samples: {nsamples}")
print(f"  Input tensor shape: {testenc.shape}")

# Show first few tokens for debugging
print(f"\nFirst 20 tokens: {testenc[0, :20].tolist()}")
print(f"Decoded: {repr(tokenizer.decode(testenc[0, :20]))}")


In [ ]:
# Configure latent quantizer
print("Configuring latent quantizer...")
configure_latent_quantizer(
    model, 
    n_bits=CONFIG['lt_bits'],
    group_size=CONFIG['lt_group_size'],
    sym=CONFIG['lt_sym'],
    clip_ratio=CONFIG['lt_clip_ratio'],
    hadamard=CONFIG['lt_hadamard']
)
print("Quantizer configured!")

# Check attention modules for debug info
print("\nChecking attention modules:")
attention_modules = []
for name, module in model.named_modules():
    if 'self_attn' in name and hasattr(module, 'rope_in_latent'):
        attention_modules.append((name, module.rope_in_latent))
        
if attention_modules:
    print(f"Found {len(attention_modules)} attention modules with rope_in_latent:")
    for name, rope_in_latent in attention_modules[:3]:  # Show first 3
        print(f"  {name}: rope_in_latent = {rope_in_latent}")
        
    # Check the first attention module in detail
    first_attn = None
    for name, module in model.named_modules():
        if 'layers.0.self_attn' in name and name.endswith('self_attn'):
            first_attn = module
            break
    
    if first_attn:
        print(f"\nFirst attention module details:")
        attrs_to_check = ['rope_in_latent', 'group_size', 'num_groups', 'total_rank_k', 'total_rank_v']
        for attr in attrs_to_check:
            if hasattr(first_attn, attr):
                value = getattr(first_attn, attr)
                print(f"  {attr}: {value}")
            else:
                print(f"  {attr}: NOT FOUND")
else:
    print("No attention modules with rope_in_latent found.")


In [ ]:
# Run evaluation with debugging
print(f"Starting PPL evaluation...")
print(f"Model: {CONFIG['model_name_or_path']}")
print(f"Dataset: {CONFIG['datasets']}")
print(f"Sequence length: {CONFIG['seqlen']}")
print(f"Device: {CONFIG['device']}")

# Run on a subset first for debugging
debug_samples = 10  # Start with just 10 samples
ppl, nlls, detailed_stats = eval_ppl_debug(
    model, tokenizer, testenc, 
    seqlen=CONFIG['seqlen'], 
    device=CONFIG['device'],
    max_samples=debug_samples,
    verbose=True
)

print(f"\n=== RESULTS ==")
print(f"Perplexity (first {debug_samples} samples): {ppl:.4f}")

# Memory check
if torch.cuda.is_available():
    print(f"\nGPU Memory after evaluation:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"  Cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


## Results Analysis and Visualization


In [ ]:
# Analyze the results
nll_values = [nll.item() for nll in nlls]

print("NLL Statistics:")
print(f"  Mean: {np.mean(nll_values):.4f}")
print(f"  Std: {np.std(nll_values):.4f}")
print(f"  Min: {np.min(nll_values):.4f}")
print(f"  Max: {np.max(nll_values):.4f}")

# Plot NLL distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(nll_values, 'b-', alpha=0.7, marker='o')
plt.title('NLL per Sample')
plt.xlabel('Sample Index')
plt.ylabel('Negative Log Likelihood')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(nll_values, bins=min(10, len(nll_values)), alpha=0.7, edgecolor='black')
plt.title('NLL Distribution')
plt.xlabel('Negative Log Likelihood')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show detailed statistics if available
if detailed_stats:
    print("\nDetailed Statistics:")
    df_stats = pd.DataFrame(detailed_stats)
    print(df_stats)
    
    # Plot logits statistics
    plt.figure(figsize=(12, 3))
    
    plt.subplot(1, 3, 1)
    plt.plot(df_stats['logits_mean'], 'g-o')
    plt.title('Logits Mean')
    plt.xlabel('Sample')
    plt.ylabel('Mean Value')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 3, 2)
    plt.plot(df_stats['logits_std'], 'r-o')
    plt.title('Logits Std')
    plt.xlabel('Sample')
    plt.ylabel('Std Value')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 3, 3)
    plt.plot(df_stats['hidden_states_norm'], 'm-o')
    plt.title('Hidden States Norm')
    plt.xlabel('Sample')
    plt.ylabel('Norm Value')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


## Full Evaluation (Optional)


In [ ]:
# Run full evaluation if needed
run_full_eval = False  # Set to True if you want to run on all samples

if run_full_eval:
    print("Running full evaluation on all samples...")
    print(f"This will process {nsamples} samples total...")
    
    full_ppl, full_nlls, _ = eval_ppl_debug(
        model, tokenizer, testenc,
        seqlen=CONFIG['seqlen'],
        device=CONFIG['device'],
        max_samples=None,  # All samples
        verbose=False
    )
    print(f"\n=== FULL RESULTS ===")
    print(f"Final Perplexity: {full_ppl:.4f}")
    
    # Compare with debug results
    print(f"\nComparison:")
    print(f"  Debug PPL ({debug_samples} samples): {ppl:.4f}")
    print(f"  Full PPL ({nsamples} samples): {full_ppl:.4f}")
    print(f"  Difference: {abs(full_ppl - ppl):.4f}")
    
else:
    print("Skipping full evaluation. Set run_full_eval=True to run on all samples.")
    print(f"This would process {nsamples} total samples.")

print("\n=== DEBUGGING COMPLETE ===")
print("To modify settings, change the CONFIG dictionary and rerun the relevant cells.")
